# Allergies : profils IgE et symptômes cutanés — soumission de départ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ml-arena/competition-baseline/blob/main/allergies/submission_baseline.ipynb)

Baseline minimale, exécutable de bout en bout, pour
**[#176 Allergies : profils IgE et symptômes cutanés](https://ml-arena.com/viewcompetition/176)**
(démonstration *Open Data University × ml-arena*).

C'est une **compétition de type fichier** : on n'envoie pas de code d'agent, mais un
`submission.csv`. Ce notebook fait le tour complet — télécharger les données, écrire une
soumission **constante** (la même probabilité pour tout le monde), l'envoyer par l'API, lire
son score — pour que vous ayez quelque chose qui tourne avant de chercher à faire mieux.

> Une soumission constante vaut **ROC AUC = 0,500** (le hasard). C'est le point de départ,
> pas un objectif : le benchmark est à 0,644 et une forêt aléatoire atteint 0,813.
> La dernière section montre comment y arriver.

## 0. Connexion

In [ ]:
!pip install -q "mlarena-sdk==0.3.0" pandas   # le paquet s'importe sous le nom `mlarena`

import mlarena
import pandas as pd

API_TOKEN = "mlk_user_REMPLACEZ_MOI"   # <-- votre jeton : ml-arena.com -> Profil -> API Keys
COMPETITION_ID = 176

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")
print(client.competition(COMPETITION_ID)["name"])

## 1. Récupérer les données

`download_dataset` écrit tous les fichiers publics de l'onglet **Datasets** dans `data/` :
`train.csv` (1 187 patients étiquetés), `test.csv` (639 patients à prédire),
`sample_submission.csv`, le dictionnaire des variables, et le notebook complet.

In [ ]:
client.download_dataset(COMPETITION_ID, "data/")

test = pd.read_csv("data/test.csv")
print(test.shape)
test[["patient_id"]].head(3)

## 2. La soumission la plus simple : une constante

Le format attendu est exactement deux colonnes et exactement les 639 identifiants de
`test.csv`, dans n'importe quel ordre, sans doublon :

```csv
patient_id,skin_symptoms
TE0000,0.5
TE0001,0.5
```

`skin_symptoms` est une **probabilité** (un réel entre 0 et 1) ; pour le ROC AUC seul
l'**ordre** des valeurs compte. Donner la même valeur à tout le monde, c'est ne donner
aucun ordre : le score sera 0,500.

In [ ]:
submission = pd.DataFrame({
    "patient_id": test["patient_id"],
    "skin_symptoms": 0.5,          # la même prédiction pour tous les patients
})
submission.to_csv("submission.csv", index=False)

assert len(submission) == len(test) and submission["patient_id"].is_unique
submission.head(3)

## 3. Envoyer

`submit` crée la soumission, téléverse le fichier et la déploie. Le fichier doit
s'appeler **`submission.csv`**. Le scoring tourne dans un conteneur : comptez une à
deux minutes.

In [ ]:
result = client.submit(competition_id=COMPETITION_ID, files=["submission.csv"])
print(result)

# Suit l'état jusqu'à la fin (deploy -> queue -> run -> noté).
for line in client.tail_logs(COMPETITION_ID, result["attache_agent_id"]):
    print(line)

## 4. Votre score et le classement

In [ ]:
print(client.status())            # MeanReward = le ROC AUC de votre soumission

client.leaderboard(COMPETITION_ID)

## 5. Faire mieux qu'une constante

Remplacez la cellule 2 par un vrai modèle. Version courte, qui vaut ≈ 0,81 en
validation croisée :

```python
from sklearn.ensemble import RandomForestClassifier

train = pd.read_csv("data/train.csv")
TARGET = "skin_symptoms"
CATEG = ["Chip_Type", "Gender", "French_Region", "Rural_or_urban_area",
         "French_Residence_Department", "Blood_Month_sample"]

y = train[TARGET].values
X_all = pd.concat([train.drop(columns=[TARGET]), test], keys=["train", "test"])
X_all = X_all.drop(columns=["patient_id"])
ige = X_all.drop(columns=CATEG).apply(pd.to_numeric, errors="coerce")
cat = pd.get_dummies(X_all[CATEG].astype(str), dummy_na=True)
X_all = pd.concat([ige, cat], axis=1).astype(float).fillna(0.0)

model = RandomForestClassifier(n_estimators=500, random_state=0, n_jobs=-1)
model.fit(X_all.loc["train"], y)
proba = model.predict_proba(X_all.loc["test"])[:, 1]

submission = pd.DataFrame({"patient_id": test["patient_id"], "skin_symptoms": proba})
submission.to_csv("submission.csv", index=False)
```

Puis, dans l'ordre de rentabilité :

* **Distinguer « non mesuré » de « négatif »** — une cellule IgE vide veut dire que la puce
  ne mesurait pas ce composant, pas que le patient est négatif. `Chip_Type` explique
  l'essentiel du motif de valeurs manquantes ; ajoutez des indicatrices de mesure au lieu
  de remplir par 0.
* **Regrouper les 241 IgE par famille** (acariens, moisissures, pollens d'arbres, graminées,
  herbacées, aliments, venins) plutôt que de les traiter indépendamment.
* **Passer les IgE au log** (`np.log1p`) : les valeurs s'étalent sur plusieurs décades.
* **Compter les co-sensibilisations** : mono- vs polysensibilisation est un marqueur
  clinique connu.
* **Gradient boosting** (`HistGradientBoostingClassifier`, XGBoost, LightGBM) gère les
  valeurs manquantes nativement.

⚠️ Le jeu de test ne fait que 639 patients : l'incertitude sur l'AUC est d'environ
**± 0,016**. Validez en croisé sur `train.csv`, ne réglez pas vos hyperparamètres sur le
classement.

Repères sur ce découpage : constante `0,500` · benchmark (nombre d'IgE positives) `0,644` ·
régression logistique `0,785` · gradient boosting `0,795` · forêt aléatoire `0,813`.